# 🏆 GoAI – Pipeline Huấn luyện trên Google Colab
**Hướng dẫn sử dụng:**
1. Bấm **Runtime → Change runtime type → T4 GPU** trước khi chạy.
2. Chuẩn bị thư mục trên Google Drive của bạn theo đúng cấu trúc:
```
MyDrive/
└── go-ai-project/
    └── backend_ai/          ← Upload toàn bộ thư mục backend_ai lên đây
        ├── data/
        │   ├── sgf_9x9/     ← Cho file .sgf 9x9 vào đây
        │   ├── sgf_13x13/   ← Cho file .sgf 13x13 vào đây
        │   └── sgf_19x19/   ← Cho file .sgf 19x19 vào đây
        ├── models/
        ├── utils/
        ├── mcts/
        └── weights/         ← File .pth sẽ được lưu tại đây sau khi train xong
```
3. Chạy từng ô (cell) theo thứ tự từ trên xuống dưới.

> **Ghi chú:** File não bộ `.pth` sau khi train xong sẽ tự lưu thẳng vào Google Drive, không cần lo bị mất khi Colab timeout.

## Bước 1: Kết nối Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive đã kết nối thành công!")

## Bước 2: Cài đặt thư viện & Cấu hình đường dẫn
> ⚠️ **Sửa biến `PROJECT_PATH` bên dưới** cho khớp với đường dẫn thư mục `backend_ai` trên Drive của bạn.

In [ ]:
import subprocess, sys

# ============================================================
# ⚙️  SỬA ĐƯỜNG DẪN NÀY CHO KHỚP VỚI DRIVE CỦA BẠN
PROJECT_PATH = '/content/drive/MyDrive/go-ai-project/backend_ai'
# ============================================================

# Cài đặt thư viện cần thiết
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torch', 'torchvision', 'numpy', '--quiet'])

# Thêm thư mục project vào Python path để import được các module nội bộ
import sys, os
sys.path.insert(0, PROJECT_PATH)
os.chdir(PROJECT_PATH)

import torch
print(f"✅ Đã cấu hình xong.")
print(f"📂 Thư mục làm việc: {os.getcwd()}")
print(f"🔥 GPU: {'CÓ (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'KHÔNG (Đang dùng CPU - Hãy bật T4 GPU!)'}") 

## Bước 3: Kiểm tra dữ liệu SGF

In [ ]:
import glob

data_dirs = {
    9:  os.path.join(PROJECT_PATH, 'data/sgf_9x9'),
    13: os.path.join(PROJECT_PATH, 'data/sgf_13x13'),
    19: os.path.join(PROJECT_PATH, 'data/sgf_19x19'),
}

print("📊 Thống kê dữ liệu ván đấu:")
print("-" * 45)
for size, d in data_dirs.items():
    files = glob.glob(os.path.join(d, '*.sgf')) if os.path.isdir(d) else []
    status = f"{len(files):,} file .sgf" if files else "⚠️  KHÔNG TÌM THẤY"
    print(f"  Bàn {size:2d}x{size:2d}  →  {status}")
print("-" * 45)
print("\n💡 Nếu thiếu dữ liệu, hãy tải file .sgf từ:")
print("   https://www.u-go.net/gamerecords/  (miễn phí, hơn 100,000 ván)")
print("   https://homepages.cwi.nl/~aeb/go/games/  (9x9)")

## Bước 4: Định nghĩa kiến trúc Model (ResNet DualCNN)
> Đây là kiến trúc nâng cấp với **ResNet blocks** và **BatchNorm**, mạnh hơn nhiều so với CNN đơn giản thông thường.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ResNetBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + residual)


class DualCNN(nn.Module):
    """
    Kiến trúc ResNet AlphaZero style.
    - channels=128, num_res_blocks=10  →  Mạnh, train ~2-4h trên T4 GPU
    - channels=64,  num_res_blocks=5   →  Nhẹ hơn, train ~45ph (vẫn khá mạnh)
    """
    def __init__(self, board_size=19, in_channels=3, channels=128, num_res_blocks=10):
        super().__init__()
        self.board_size = board_size

        self.conv_in = nn.Conv2d(in_channels, channels, 3, padding=1, bias=False)
        self.bn_in   = nn.BatchNorm2d(channels)
        self.res_blocks = nn.Sequential(*[ResNetBlock(channels) for _ in range(num_res_blocks)])

        # Policy head
        self.policy_conv = nn.Conv2d(channels, 2, 1, bias=False)
        self.policy_bn   = nn.BatchNorm2d(2)
        self.policy_fc   = nn.Linear(2 * board_size * board_size, board_size * board_size)

        # Value head
        self.value_conv = nn.Conv2d(channels, 1, 1, bias=False)
        self.value_bn   = nn.BatchNorm2d(1)
        self.value_fc1  = nn.Linear(board_size * board_size, 256)
        self.value_fc2  = nn.Linear(256, 1)

    def forward(self, x):
        x = F.relu(self.bn_in(self.conv_in(x)))
        x = self.res_blocks(x)

        p = F.relu(self.policy_bn(self.policy_conv(x))).view(x.size(0), -1)
        policy_out = F.softmax(self.policy_fc(p), dim=1)

        v = F.relu(self.value_bn(self.value_conv(x))).view(x.size(0), -1)
        v = F.relu(self.value_fc1(v))
        value_out = torch.tanh(self.value_fc2(v))

        return policy_out, value_out

print("✅ Kiến trúc DualCNN (ResNet) đã sẵn sàng.")

## Bước 5: Dataset & Vòng lặp huấn luyện

In [ ]:
import re, glob, time
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

# Import module từ project
from models.encoder import BoardEncoder
from utils.game_logic import GoState, BLACK, WHITE


# ── Hàm đọc SGF không cần thư viện sgf ngoài ────────────────────────────────
def parse_sgf(file_path: str, board_size: int):
    """Đọc file SGF, trả về (list_of_moves, winner: 1|-1)"""
    try:
        content = open(file_path, encoding='utf-8', errors='ignore').read()
    except Exception:
        return [], BLACK

    # Xác định người thắng từ tag RE[...]
    winner = BLACK
    m = re.search(r'RE\[([BW])', content, re.IGNORECASE)
    if m:
        winner = BLACK if m.group(1).upper() == 'B' else WHITE

    moves = []
    for coord in re.findall(r';[BW]\[([a-z]{0,2})\]', content):
        if not coord or len(coord) != 2:
            moves.append(None)
            continue
        c = ord(coord[0]) - ord('a')
        r = ord(coord[1]) - ord('a')
        if 0 <= r < board_size and 0 <= c < board_size:
            moves.append((r, c))
        else:
            moves.append(None)
    return moves, winner


# ── Dataset ──────────────────────────────────────────────────────────────────
class SGFDataset(Dataset):
    def __init__(self, sgf_paths, board_size, max_files=None):
        self.samples = []
        self.encoder = BoardEncoder()
        paths = sgf_paths[:max_files] if max_files else sgf_paths
        self._build(paths, board_size)

    def _build(self, paths, board_size):
        ok = 0
        for path in paths:
            moves, winner = parse_sgf(path, board_size)
            if not moves:
                continue
            state = GoState(size=board_size)
            for move in moves:
                enc = self.encoder.encode(state)
                # Policy target
                if move is None:
                    pidx = board_size * board_size  # pass
                else:
                    pidx = move[0] * board_size + move[1]
                pidx = min(pidx, board_size * board_size - 1)
                # Value target – dựa vào người thắng thật từ SGF
                vtgt = 1.0 if state.current_player == winner else -1.0
                self.samples.append((enc, pidx, vtgt))
                if not state.apply_move(move):
                    break
            ok += 1
        print(f"  ✅ Đọc xong {ok}/{len(paths)} ván  →  {len(self.samples):,} mẫu dữ liệu")

    def __len__(self):  return len(self.samples)
    def __getitem__(self, i):
        b, p, v = self.samples[i]
        return (torch.tensor(b, dtype=torch.float32),
                torch.tensor(p, dtype=torch.long),
                torch.tensor([v], dtype=torch.float32))


# ── Vòng lặp train 1 kích thước ──────────────────────────────────────────────
def train_one_size(
    board_size: int,
    data_dir:   str,
    weights_dir: str,
    epochs:      int   = 30,
    batch_size:  int   = 128,
    lr:          float = 1e-3,
    channels:    int   = 128,
    num_res_blocks: int = 10,
    max_files:   int   = None,
):
    print(f"\n{'='*55}")
    print(f"🎯 TRAINING BÀN CỜ {board_size}x{board_size}")
    print(f"   Epochs={epochs}  Batch={batch_size}  LR={lr}")
    print(f"   Channels={channels}  ResBlocks={num_res_blocks}")
    print(f"{'='*55}")

    sgf_paths = glob.glob(os.path.join(data_dir, '*.sgf'))
    if not sgf_paths:
        print(f"  ⚠️  Không có file .sgf trong {data_dir} — bỏ qua.")
        return

    print(f"  📂 {len(sgf_paths):,} ván cờ tìm thấy")
    dataset = SGFDataset(sgf_paths, board_size, max_files=max_files)
    if len(dataset) == 0:
        print("  ⚠️  Dataset rỗng — bỏ qua.")
        return

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                        drop_last=True, num_workers=2, pin_memory=True)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model  = DualCNN(board_size=board_size, channels=channels,
                     num_res_blocks=num_res_blocks).to(device)

    optimizer = Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)
    p_loss_fn = nn.CrossEntropyLoss()
    v_loss_fn = nn.MSELoss()

    best_loss = float('inf')
    os.makedirs(weights_dir, exist_ok=True)
    save_path = os.path.join(weights_dir, f'dualcnn_{board_size}x{board_size}_final.pth')

    model.train()
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        total_loss = p_total = v_total = 0.0

        for boards, ptgt, vtgt in loader:
            boards, ptgt, vtgt = boards.to(device), ptgt.to(device), vtgt.to(device)
            optimizer.zero_grad()
            p_pred, v_pred = model(boards)
            loss = p_loss_fn(p_pred, ptgt) + v_loss_fn(v_pred, vtgt)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Chống gradient bùng nổ
            optimizer.step()

            total_loss += loss.item()
            p_total    += p_loss_fn(p_pred, ptgt).item()
            v_total    += v_loss_fn(v_pred, vtgt).item()

        scheduler.step()
        avg   = total_loss / len(loader)
        elapsed = time.time() - t0
        print(f"  Epoch [{epoch:03d}/{epochs}]  Loss={avg:.4f}  "
              f"(P={p_total/len(loader):.4f} V={v_total/len(loader):.4f})  "
              f"LR={scheduler.get_last_lr()[0]:.6f}  {elapsed:.0f}s")

        # Lưu checkpoint tốt nhất
        if avg < best_loss:
            best_loss = avg
            torch.save(model.state_dict(), save_path)
            print(f"  💾 Checkpoint mới (loss={best_loss:.4f}) → {save_path}")

    print(f"\n🏁 Kết thúc training {board_size}x{board_size}. Não bộ tốt nhất: {save_path}")

print("✅ Đã định nghĩa xong Dataset + Vòng lặp train.")

## Bước 6: ⚙️ Cấu hình tham số Train
> **Hướng dẫn chỉnh sửa tham số:**
> - `epochs`: Nhiều epochs hơn → AI học sâu hơn nhưng mất nhiều thời gian hơn.
> - `channels` + `num_res_blocks`: Số càng lớn → AI càng thông minh nhưng cần GPU mạnh hơn.
> - `max_files`: Giới hạn số ván dùng để train (để test nhanh trước khi train full).

In [ ]:
# ====================================================
# ⚙️  THAY ĐỔI CÁC THAM SỐ Ở ĐÂY
# ====================================================
TRAIN_CONFIG = {
    9: dict(
        epochs        = 40,     # Bàn 9x9 train nhanh hơn, có thể nhiều epoch hơn
        batch_size    = 256,
        lr            = 1e-3,
        channels      = 128,
        num_res_blocks= 10,
        max_files     = None,   # None = dùng tất cả file
    ),
    13: dict(
        epochs        = 30,
        batch_size    = 128,
        lr            = 1e-3,
        channels      = 128,
        num_res_blocks= 10,
        max_files     = None,
    ),
    19: dict(
        epochs        = 30,     # 19x19 nặng hơn, tốn nhiều thời gian nhất
        batch_size    = 64,
        lr            = 1e-3,
        channels      = 128,
        num_res_blocks= 10,
        max_files     = None,
    ),
}

WEIGHTS_DIR = os.path.join(PROJECT_PATH, 'weights')
print("✅ Cấu hình tham số đã sẵn sàng.")
print(f"📁 Weights sẽ lưu tại: {WEIGHTS_DIR}")

## Bước 7: 🚀 Bắt đầu Train toàn bộ 3 kích thước bàn cờ
> Ô này sẽ tự động train lần lượt **9x9 → 13x13 → 19x19**. Sau mỗi kích thước, file `.pth` được lưu thẳng vào Google Drive.

In [ ]:
import time

start_all = time.time()
print("🚀 BẮT ĐẦU TOÀN BỘ QUY TRÌNH HUẤN LUYỆN")
print(f"⏱️  {time.strftime('%H:%M:%S')}\n")

for board_size in [9, 13, 19]:
    data_dir = data_dirs[board_size]
    cfg = TRAIN_CONFIG[board_size]
    train_one_size(
        board_size    = board_size,
        data_dir      = data_dir,
        weights_dir   = WEIGHTS_DIR,
        **cfg
    )

elapsed_all = time.time() - start_all
h, m = divmod(int(elapsed_all), 3600)
m, s = divmod(m, 60)
print(f"\n{'='*55}")
print(f"🎉 HOÀN TẤT TOÀN BỘ! Tổng thời gian: {h}h {m}m {s}s")
print(f"📁 Các file não bộ đã lưu tại: {WEIGHTS_DIR}")
print(f"{'='*55}")

## Bước 8: Kiểm tra kết quả
> Ô này liệt kê các file `weights` đã train xong và cho bạn test thử model.

In [ ]:
import os

print("📦 Các file 'não bộ' đã tạo ra:")
print("-" * 50)
for fname in sorted(os.listdir(WEIGHTS_DIR)):
    if fname.endswith('.pth'):
        fpath = os.path.join(WEIGHTS_DIR, fname)
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"  ✅ {fname}  ({size_mb:.1f} MB)")

print("\n🧪 Test thử model 9x9...")
try:
    m = DualCNN(board_size=9, channels=128, num_res_blocks=10)
    m.load_state_dict(torch.load(os.path.join(WEIGHTS_DIR, 'dualcnn_9x9_final.pth'), map_location='cpu'))
    m.eval()
    with torch.no_grad():
        dummy = torch.zeros(1, 3, 9, 9)
        policy, value = m(dummy)
    print(f"  Policy shape: {policy.shape}  ✅")
    print(f"  Value (bàn trống): {value.item():.4f}  (gần 0 là bình thường)")
    print("\n🏆 Model đã sẵn sàng sử dụng!")
    print("💡 Tải file .pth về máy, chép vào thư mục backend_ai/weights/ là xong!")
except Exception as e:
    print(f"  ❌ Lỗi khi test: {e}")